# Chronos-2 uncertainty-regularised LoRA: M6 inference

Runs the three uncertainty-regularised checkpoints from
`chronos2_uncertainty_lora_training.ipynb` over the 12 official M6 rounds, and
saves the raw quantile forecasts.

| | |
|---|---|
| Models | `amazon/chronos-2` plus one uncertainty-regularised LoRA adapter per beta |
| Betas | 0.5, 1.0, 2.0, processed sequentially |
| Fixed | context 512, horizon 20, `cross_learning=False`, official M6 asset order |
| Input | the same 12 shared round-context CSVs every other model used |
| Output | 12 quantile NPZ files per beta, in separate folders |

The inference path is identical to the ordinary Chronos-2 LoRA run, so the only
difference between those forecasts and these is the training objective.

No scenario generation, no M6 probability conversion and no RPS: those are done
locally afterwards with the project's existing post-processing.

Run the training notebook first. This one stops if a checkpoint is missing.

## 1. Install Chronos-2 and PEFT

In [ ]:
%pip install -q "chronos-forecasting==2.3.1" "transformers>=4.41,<5" "peft>=0.11"

In [ ]:
%pip uninstall -y torchao

## 2. Imports and settings

Identical to the ordinary Chronos-2 LoRA inference run.

In [ ]:
import json
import traceback
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from chronos import Chronos2Pipeline

BASE_MODEL_ID = "amazon/chronos-2"

CONTEXT_LENGTH = 512
PREDICTION_LENGTH = 20
RANDOM_SEED = 42
ROUNDS = range(1, 13)
BETAS = [0.5, 1.0, 2.0]

# Chronos-2 shares information across the variates of ONE task. cross_learning
# shares information across SEPARATE tasks, and this submits a single task, so
# it stays False exactly as in every previous Chronos-2 run.
CROSS_LEARNING = False

EXPECTED_QUANTILE_LEVELS = (
    [0.01, 0.05] + [round(0.05 * i, 2) for i in range(2, 19)] + [0.95, 0.99]
)

ROUND_SCHEDULE = {
    1:  ("2020-03-19", "2022-03-04", "2022-03-07", "2022-04-01"),
    2:  ("2020-04-16", "2022-04-01", "2022-04-04", "2022-04-29"),
    3:  ("2020-05-14", "2022-04-29", "2022-05-02", "2022-05-27"),
    4:  ("2020-06-11", "2022-05-27", "2022-05-30", "2022-06-24"),
    5:  ("2020-07-09", "2022-06-24", "2022-06-27", "2022-07-22"),
    6:  ("2020-08-06", "2022-07-22", "2022-07-25", "2022-08-19"),
    7:  ("2020-09-03", "2022-08-19", "2022-08-22", "2022-09-16"),
    8:  ("2020-10-01", "2022-09-16", "2022-09-19", "2022-10-14"),
    9:  ("2020-10-29", "2022-10-14", "2022-10-17", "2022-11-11"),
    10: ("2020-11-26", "2022-11-11", "2022-11-14", "2022-12-09"),
    11: ("2020-12-24", "2022-12-09", "2022-12-12", "2023-01-06"),
    12: ("2021-01-21", "2023-01-06", "2023-01-09", "2023-02-03"),
}

# Genuine leading missing history, preserved as NaN for the model's own masking.
EXPECTED_LEADING_NAN = {r: {"OGN": 302 - 20 * (r - 1)} for r in ROUNDS}
EXPECTED_LEADING_NAN[1] = {"CARR": 1, "OGN": 302}

OFFICIAL_ASSET_ORDER = [
    "ABBV", "ACN", "AEP", "AIZ", "ALLE", "AMAT", "AMP", "AMZN", "AVB", "AVY",
    "AXP", "BDX", "BF-B", "BMY", "BR", "CARR", "CDW", "CE", "CHTR", "CNC",
    "CNP", "COP", "CTAS", "CZR", "DG", "DPZ", "DRE", "DXC", "EWA", "EWC",
    "EWG", "EWH", "EWJ", "EWL", "EWQ", "EWT", "EWU", "EWY", "EWZ", "FTV",
    "GOOG", "GPC", "GSG", "HIG", "HIGH.L", "HST", "HYG", "IAU", "ICLN",
    "IEAA.L", "IEF", "IEFM.L", "IEMG", "IEUS", "IEVL.L", "IGF", "INDA",
    "IUMO.L", "IUVL.L", "IVV", "IWM", "IXN", "JPEA.L", "JPM", "KR", "LQD",
    "MCHI", "META", "MVEU.L", "OGN", "PG", "PPL", "PRU", "PYPL", "RE",
    "REET", "ROL", "ROST", "SEGA.L", "SHY", "SLV", "SPMV.L", "TLT", "UNH",
    "URI", "V", "VRSK", "VXX", "WRK", "XLB", "XLC", "XLE", "XLF", "XLI",
    "XLK", "XLP", "XLU", "XLV", "XLY", "XOM",
]
N_ASSETS = len(OFFICIAL_ASSET_ORDER)

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print("Versions")
print(f"  chronos-forecasting : {version('chronos-forecasting')}")
print(f"  transformers        : {version('transformers')}")
print(f"  torch               : {torch.__version__}")
print(f"\nUncertainty-regularised LoRA, betas {BETAS}, "
      f"M6 rounds {min(ROUNDS)}-{max(ROUNDS)}")
print(f"Total files expected : {len(BETAS)} x {len(list(ROUNDS))} = "
      f"{len(BETAS) * len(list(ROUNDS))}")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Paths

Contexts come from the same shared folder every model has used. Each beta writes
to its own output folder, so no configuration can overwrite another or the
ordinary-LoRA outputs.

In [ ]:
# Existing shared context folder - the same 12 files as every previous run.
DRIVE_CONTEXT_DIR = Path("/content/drive/MyDrive/HonoursResearch/Round_1_Context")

CHECKPOINT_ROOT = Path("/content/drive/MyDrive/HonoursResearch/checkpoints/chronos2_uncertainty_lora")
OUTPUT_ROOT = Path("/content/drive/MyDrive/HonoursResearch/outputs/chronos2_uncertainty_lora")

OUTPUT_PREFIX = "chronos2_uncertainty_lora"


def checkpoint_dir(beta: float) -> Path:
    return CHECKPOINT_ROOT / f"beta{beta}"


def output_dir(beta: float) -> Path:
    return OUTPUT_ROOT / f"beta{beta}" / "m6"


def context_path(round_number: int) -> Path:
    return DRIVE_CONTEXT_DIR / f"round_{round_number:02d}_context.csv"


def quantiles_path(beta: float, round_number: int) -> Path:
    return output_dir(beta) / (f"{OUTPUT_PREFIX}_beta{beta}"
                               f"_round{round_number:02d}_quantiles.npz")


if not DRIVE_CONTEXT_DIR.is_dir():
    raise FileNotFoundError(
        f"Context folder not found:\n  {DRIVE_CONTEXT_DIR}\n"
        "It must be the SAME folder used by the earlier model notebooks."
    )

missing_checkpoints = [b for b in BETAS
                       if not checkpoint_dir(b).is_dir()
                       or not any(checkpoint_dir(b).iterdir())]
if missing_checkpoints:
    raise FileNotFoundError(
        f"No fine-tuned checkpoint for beta {missing_checkpoints}:\n  "
        + f"\n  ".join(str(checkpoint_dir(b)) for b in missing_checkpoints)
        + f"\n\nRun chronos2_uncertainty_lora_training.ipynb to completion "
        "first. This notebook only runs inference and cannot create them."
    )

for beta in BETAS:
    output_dir(beta).mkdir(parents=True, exist_ok=True)

print(f"Contexts (existing, shared) : {DRIVE_CONTEXT_DIR}")
for beta in BETAS:
    print(f"  beta {beta}: {checkpoint_dir(beta)}")
    print(f"           -> {output_dir(beta)}")
print(f"\nExample output filename : {quantiles_path(BETAS[0], 1).name}")

## 5. Check the 12 context files are present

In [ ]:
missing = [context_path(r).name for r in ROUNDS if not context_path(r).is_file()]
if missing:
    raise FileNotFoundError(
        f"Missing context files in {DRIVE_CONTEXT_DIR}:\n  " + f"\n  ".join(missing)
        + f"\nThey must be the same round_NN_context.csv files used by every "
          "other model, copied from Data/contexts/."
    )
print(f"All {len(list(ROUNDS))} context files present in {DRIVE_CONTEXT_DIR}")

## 6. GPU check

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. Runtime > Change runtime type > Hardware "
        "accelerator: GPU, then re-run from section 3."
    )
GPU_NAME = torch.cuda.get_device_name(0)
print(f"GPU: {GPU_NAME}")

## 7. Helpers

The same context validation, forecasting and save/verify logic as the ordinary
Chronos-2 LoRA inference notebook, with the beta carried through so that no two
configurations can share a file.

In [ ]:
def load_and_validate_context(round_number: int):
    """Load one M6 context, validate it, and build the model input."""
    ctx_start, origin, fc_start, fc_end = ROUND_SCHEDULE[round_number]
    df = pd.read_csv(context_path(round_number), parse_dates=["date"])

    assert df.shape[0] == CONTEXT_LENGTH, (
        f"Round {round_number}: expected {CONTEXT_LENGTH} rows, found {df.shape[0]}"
    )
    assert list(df.columns) == ["date"] + OFFICIAL_ASSET_ORDER, (
        f"Round {round_number}: columns are not 'date' + the official M6 asset order"
    )
    dates = df["date"]
    assert dates.is_monotonic_increasing, f"Round {round_number}: dates not ascending"
    assert not dates.duplicated().any(), f"Round {round_number}: duplicate dates"
    assert dates.iloc[0] == pd.Timestamp(ctx_start), f"Round {round_number}: context start"
    assert dates.iloc[-1] == pd.Timestamp(origin), f"Round {round_number}: context end"
    assert (dates <= pd.Timestamp(origin)).all(), (
        f"Round {round_number}: context contains a date after the origin"
    )

    matrix = df[OFFICIAL_ASSET_ORDER].to_numpy(dtype=np.float64).T
    assert matrix.shape == (N_ASSETS, CONTEXT_LENGTH)

    # Leading gaps are genuine later inceptions and stay NaN.
    leading_missing, interior_missing = {}, {}
    for i, symbol in enumerate(OFFICIAL_ASSET_ORDER):
        row = matrix[i]
        n_lead = int(np.argmax(~np.isnan(row))) if np.isnan(row[0]) else 0
        if n_lead:
            leading_missing[symbol] = n_lead
        if np.isnan(row[n_lead:]).any():
            interior_missing[symbol] = int(np.isnan(row[n_lead:]).sum())
    assert not interior_missing, (
        f"Round {round_number}: unexpected interior NaNs {interior_missing}"
    )
    assert leading_missing == EXPECTED_LEADING_NAN[round_number], (
        f"Round {round_number}: leading NaN pattern changed - "
        f"expected {EXPECTED_LEADING_NAN[round_number]}, found {leading_missing}"
    )

    forecast_dates = pd.bdate_range(fc_start, fc_end)
    assert len(forecast_dates) == PREDICTION_LENGTH
    return matrix, forecast_dates


def forecast_round(pipeline, context_matrix, round_number: int, n_quantiles: int):
    """Forecast one round as a single multivariate task.

    Input  : (1, 100, 512) - one task, 100 target variates, 512 observations.
    Output : (100, 20, 21) - assets x forecast days x native quantiles.
    """
    model_input = context_matrix[np.newaxis, :, :].astype(np.float32)
    assert model_input.shape == (1, N_ASSETS, CONTEXT_LENGTH)

    outputs = pipeline.predict(inputs=model_input,
                               prediction_length=PREDICTION_LENGTH,
                               cross_learning=CROSS_LEARNING)
    assert len(outputs) == 1
    raw = outputs[0]
    raw = raw.to(torch.float32).cpu().numpy() if torch.is_tensor(raw) else np.asarray(raw)

    # Inspect the returned layout instead of assuming it. 20 days and 21
    # quantiles differ in size, so the axes are unambiguous.
    if raw.shape == (N_ASSETS, n_quantiles, PREDICTION_LENGTH):
        quantile_forecasts = np.transpose(raw, (0, 2, 1))
    elif raw.shape == (N_ASSETS, PREDICTION_LENGTH, n_quantiles):
        quantile_forecasts = raw
    else:
        raise ValueError(
            f"Round {round_number}: unrecognised output shape {raw.shape}"
        )

    quantile_forecasts = np.ascontiguousarray(quantile_forecasts)
    assert quantile_forecasts.shape == (N_ASSETS, PREDICTION_LENGTH, n_quantiles)
    assert np.isfinite(quantile_forecasts).all(), (
        f"Round {round_number}: forecasts contain NaN or infinite values"
    )

    # Crossings are reported, never silently corrected.
    crossings = int((np.diff(quantile_forecasts, axis=2) < -1e-6).sum())
    return quantile_forecasts, crossings


def save_and_verify_round(beta, round_number, quantile_forecasts, forecast_dates,
                          quantile_levels, crossings, training_metadata):
    """Save the raw quantile forecasts, then reload and verify the file."""
    path = quantiles_path(beta, round_number)
    _, origin, fc_start, fc_end = ROUND_SCHEDULE[round_number]
    date_strings = np.array([d.strftime("%Y-%m-%d") for d in forecast_dates])
    n_quantiles = int(quantile_levels.size)

    run_metadata = {
        "base_model_id": BASE_MODEL_ID,
        "adapter": "uncertainty_regularised_lora",
        "beta": beta,
        "checkpoint": str(checkpoint_dir(beta)),
        "round": round_number,
        "origin": origin,
        "forecast_start": fc_start,
        "forecast_end": fc_end,
        "context_length": CONTEXT_LENGTH,
        "prediction_length": PREDICTION_LENGTH,
        "cross_learning": CROSS_LEARNING,
        "random_seed": RANDOM_SEED,
        "quantile_crossings": crossings,
        "coverage_targets": training_metadata.get("coverage_targets"),
        "coverage_smoothing": training_metadata.get("coverage_smoothing"),
        "lora_rank": training_metadata.get("lora_rank"),
        "lora_alpha": training_metadata.get("lora_alpha"),
        "max_steps": training_metadata.get("max_steps"),
        "generated_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"),
    }

    np.savez_compressed(
        path,
        quantile_forecasts=quantile_forecasts,
        quantile_levels=quantile_levels,
        asset_symbols=np.array(OFFICIAL_ASSET_ORDER),
        forecast_dates=date_strings,
        run_metadata=np.array(json.dumps(run_metadata)),
    )

    with np.load(path, allow_pickle=False) as reloaded:
        r_forecasts = reloaded["quantile_forecasts"]
        r_levels = reloaded["quantile_levels"]
        r_symbols = list(reloaded["asset_symbols"])
        r_dates = list(reloaded["forecast_dates"])

    assert r_forecasts.shape == (N_ASSETS, PREDICTION_LENGTH, n_quantiles)
    assert np.array_equal(r_forecasts, quantile_forecasts), "saved values differ"
    assert np.isfinite(r_forecasts).all(), "reloaded values not finite"
    assert np.allclose(r_levels, quantile_levels), "quantile levels changed"
    assert r_symbols == OFFICIAL_ASSET_ORDER, "asset ordering changed"
    assert r_dates == list(date_strings), "forecast dates changed"
    return path


def already_done(beta: float, round_number: int, n_quantiles: int) -> bool:
    """True if this beta and round is saved and passes the essential checks."""
    path = quantiles_path(beta, round_number)
    if not path.is_file():
        return False
    _, _, fc_start, fc_end = ROUND_SCHEDULE[round_number]
    expected_dates = [d.strftime("%Y-%m-%d") for d in pd.bdate_range(fc_start, fc_end)]
    try:
        with np.load(path, allow_pickle=False) as f:
            if not {"quantile_forecasts", "quantile_levels",
                    "asset_symbols", "forecast_dates"} <= set(f.files):
                return False
            if f["quantile_forecasts"].shape != (N_ASSETS, PREDICTION_LENGTH, n_quantiles):
                return False
            if list(f["asset_symbols"]) != OFFICIAL_ASSET_ORDER:
                return False
            if list(f["forecast_dates"]) != expected_dates:
                return False
            if not np.isfinite(f["quantile_forecasts"]).all():
                return False
            if "run_metadata" in f.files:
                meta = json.loads(str(f["run_metadata"]))
                if meta.get("beta") != beta:
                    return False
            return True
    except Exception:
        return False


print("Helpers defined.")

### Run every beta over all 12 rounds

Checkpoints are loaded one at a time and released before the next, so only one
model is ever on the GPU. Re-running resumes at the individual beta-and-round
level: anything already saved and verified is skipped.

In [ ]:
records, failures, skipped = [], [], []
experiment_started = datetime.now(timezone.utc)

for beta in BETAS:
    directory = checkpoint_dir(beta)
    print(f"\n{'#' * 74}\nbeta = {beta}\n{'#' * 74}")

    metadata_file = directory / "training_metadata.json"
    training_metadata = (json.loads(metadata_file.read_text(encoding="utf-8"))
                         if metadata_file.is_file() else {})
    if training_metadata:
        print(f"  trained with targets {training_metadata.get('coverage_targets')}, "
              f"smoothing {training_metadata.get('coverage_smoothing')}")

    pipeline = Chronos2Pipeline.from_pretrained(
        str(directory), device_map="cuda",
        import_allowlist=["chronos.chronos2.model"],
    )
    quantile_levels = np.asarray(pipeline.quantiles, dtype=np.float64)
    n_quantiles = int(quantile_levels.size)
    assert np.all(np.diff(quantile_levels) > 0), "quantile levels are not increasing"
    assert n_quantiles == len(EXPECTED_QUANTILE_LEVELS), (
        f"expected {len(EXPECTED_QUANTILE_LEVELS)} quantile levels, got {n_quantiles}"
    )
    print(f"  loaded, {n_quantiles} quantile levels")

    for r in ROUNDS:
        _, origin, fc_start, fc_end = ROUND_SCHEDULE[r]
        if already_done(beta, r, n_quantiles):
            skipped.append((beta, r))
            print(f"  round {r:02d}: already saved and verified, skipped")
            continue
        try:
            context_matrix, forecast_dates = load_and_validate_context(r)
            started = datetime.now(timezone.utc)
            quantile_forecasts, crossings = forecast_round(
                pipeline, context_matrix, r, n_quantiles)
            elapsed = (datetime.now(timezone.utc) - started).total_seconds()
            saved_path = save_and_verify_round(
                beta, r, quantile_forecasts, forecast_dates, quantile_levels,
                crossings, training_metadata)
            records.append({"beta": beta, "round": r, "seconds": round(elapsed, 1),
                            "crossings": crossings, "file": saved_path.name})
            flag = f"  !! {crossings} quantile crossings" if crossings else ""
            print(f"  round {r:02d}: {tuple(quantile_forecasts.shape)} in "
                  f"{elapsed:.1f}s, saved{flag}")
            del quantile_forecasts, context_matrix
        except Exception as exc:
            failures.append((beta, r, f"{type(exc).__name__}: {exc}"))
            print(f"  !! beta {beta} round {r:02d} FAILED")
            traceback.print_exc()
        torch.cuda.empty_cache()

    del pipeline
    torch.cuda.empty_cache()

total_minutes = (datetime.now(timezone.utc) - experiment_started).total_seconds() / 60
print(f"\nFinished in {total_minutes:.1f} min. Generated {len(records)}, "
      f"skipped {len(skipped)}, failed {len(failures)} of "
      f"{len(BETAS) * len(list(ROUNDS))}.")
if failures:
    print(f"FAILED: {[(b, r) for b, r, _ in failures]}")

## 8. Verify every saved file

Re-reads all files from Drive, independently of the loop above.

In [ ]:
ALL_VERIFIED = True
CROSSING_FILES = []
print(f"{'beta':>5} {'rnd':>4}  {'status':<8} {'crossings':>10}  shape")

for beta in BETAS:
    for r in ROUNDS:
        path = quantiles_path(beta, r)
        if not path.is_file():
            ALL_VERIFIED = False
            print(f"{beta:>5} {r:>4}  MISSING")
            continue
        _, _, fc_start, fc_end = ROUND_SCHEDULE[r]
        expected_dates = [d.strftime("%Y-%m-%d")
                          for d in pd.bdate_range(fc_start, fc_end)]
        with np.load(path, allow_pickle=False) as f:
            forecasts = f["quantile_forecasts"]
            levels = f["quantile_levels"]
            symbols = list(f["asset_symbols"])
            fdates = list(f["forecast_dates"])
            meta = json.loads(str(f["run_metadata"])) if "run_metadata" in f.files else {}
        crossings = int((np.diff(forecasts, axis=2) < -1e-6).sum())
        ok = (forecasts.shape == (N_ASSETS, PREDICTION_LENGTH, int(levels.size))
              and symbols == OFFICIAL_ASSET_ORDER
              and fdates == expected_dates
              and bool(np.isfinite(forecasts).all())
              and meta.get("beta") == beta)
        ALL_VERIFIED = ALL_VERIFIED and ok
        if crossings:
            CROSSING_FILES.append((beta, r, crossings))
        print(f"{beta:>5} {r:>4}  {'OK' if ok else 'FAILED':<8} "
              f"{crossings:>10}  {forecasts.shape}")

print(f"\nAll {len(BETAS) * len(list(ROUNDS))} files present and verified: "
      f"{ALL_VERIFIED}")
if CROSSING_FILES:
    print(f"Files with quantile crossings, left uncorrected: {CROSSING_FILES}")
else:
    print("No quantile crossings in any saved file.")

## 9. Completion summary

In [ ]:
print("Chronos-2 uncertainty-regularised LoRA - M6 rounds 1-12")
print(f"  base model : {BASE_MODEL_ID} plus one adapter per beta")
print(f"  betas      : {BETAS}")
print(f"  fixed      : context {CONTEXT_LENGTH}, horizon {PREDICTION_LENGTH}, "
      f"cross_learning {CROSS_LEARNING}, {N_ASSETS} assets")
print(f"  verified   : {ALL_VERIFIED}")
print("  post-processing: none here (no scenarios, no quintiles, no RPS)")
print()
for beta in BETAS:
    saved = sum(1 for r in ROUNDS if quantiles_path(beta, r).is_file())
    status = "COMPLETE" if saved == len(list(ROUNDS)) else "INCOMPLETE"
    print(f"  beta {beta}: {saved}/12 rounds {status}")
    print(f"           {output_dir(beta)}")

if all(sum(1 for r in ROUNDS if quantiles_path(b, r).is_file()) == len(list(ROUNDS))
       for b in BETAS):
    print(f"\n  All configurations COMPLETE. Copy each beta folder from Drive into")
    print("  TSFM Uncertainty Experiment/model_outputs/chronos2_uncertainty_lora/")
    print("  keeping the beta folder names, then score locally.")
else:
    print(f"\n  INCOMPLETE. Re-run the loop in section 7 to resume; finished "
          "beta/round files are skipped.")